[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiehou-lab/urban-ai/blob/main/notebooks/lab1_text_mining.ipynb)

# Lab 1: Text Mining Community Voice

**Duration:** ~1.0 hour
**TA lead:** Sean
**Course:** Urban AI — AI-Driven Decision Support for Real-World Urban Challenges (MSU AI-Ready Initiative)

## Learning goals
- Turn free-text community reviews into structured themes with TF-IDF keyword extraction.
- Estimate review sentiment with a simple, transparent lexicon method.
- Check whose voices are over- or under-represented in a text dataset relative to who lives in the city.
- Produce a top-themes table with a documented bias note.


## Before you start: Track A vs. Track B

Every Urban AI lab has two tracks. Both produce the **same artifact**: **Top-themes table + one bias note**.

- **Track A — No code (default).** Use a web word-cloud / sentiment tool on the supplied review text. No installation, no Python required — use this track if you would rather click through a web tool.
- **Track B — Colab (this notebook).** Run TF-IDF + a simple sentiment score + topic-keyword extraction over geolocated reviews. You will run pre-written cells and change only the parameters marked `# ▶ CHANGE ME`. You will never need to write code from scratch.

Both tracks end with the same 4 reflection prompts (the last cell of this notebook).


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

RNG = np.random.default_rng(7)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
print("Setup complete.")

## 2. Load the data: park & transit-stop reviews

> **Synthetic-but-realistic data.** The dataset below is generated in this notebook with a fixed
> random seed so the lab runs the same way for everyone, completely offline. It is built to look and
> behave like real urban data, but it is not real. To swap in real data for your own city, instructors
> can replace the data-generation cell with a download/load from a real source such as:
- Public review platforms (Google/Yelp reviews for parks and transit stops), collected per platform terms of service
- City engagement-survey exports or 311 "compliment/complaint" text fields
- Open transit-agency rider-survey datasets, often on city/agency open data portals


In [ ]:
sites = pd.DataFrame({
    "site_name": ["Riverside Park", "Old Town Plaza", "Eastgate Transit Hub", "Millbrook Greenway",
                  "Southside Community Park", "Uptown Rail Station", "Riverside Bike Path", "Eastgate Pocket Park"],
    "neighborhood": ["Riverside", "Old Town", "Eastgate", "Millbrook",
                      "Southside", "Uptown", "Riverside", "Eastgate"],
    "site_type": ["park", "plaza", "transit", "park", "park", "transit", "park", "park"],
})

# Each neighborhood's share of city population (synthetic, used later to check representativeness)
pop_share = {"Riverside": 0.12, "Old Town": 0.18, "Eastgate": 0.20,
             "Millbrook": 0.10, "Southside": 0.28, "Uptown": 0.12}

# Not everyone reviews at the same rate -- e.g. wealthier/well-connected neighborhoods often
# leave disproportionately more online reviews than their population share would predict.
review_volume_weight = {"Riverside": 0.12, "Old Town": 0.30, "Eastgate": 0.15,
                         "Millbrook": 0.08, "Southside": 0.15, "Uptown": 0.20}

positive_phrases = ["is clean and well maintained", "has great shade and benches",
                     "feels safe in the evening", "is easy to reach by bike",
                     "has frequent and reliable service", "is accessible for strollers and wheelchairs"]
negative_phrases = ["needs better lighting at night", "feels unsafe after dark",
                     "is often crowded and noisy", "has broken benches and litter",
                     "has unreliable and delayed service", "is hard to reach without a car"]
neutral_phrases = ["is used mostly on weekends", "could use more trash cans",
                    "has a small parking lot", "is near the elementary school"]

n_reviews = 500
site_by_nb = sites.groupby("neighborhood")["site_name"].apply(list).to_dict()
nb_list = list(review_volume_weight.keys())
chosen_nb = RNG.choice(nb_list, size=n_reviews, p=[review_volume_weight[k] for k in nb_list])
chosen_site = [RNG.choice(site_by_nb[nb]) for nb in chosen_nb]
sentiment_roll = RNG.random(n_reviews)

records = []
for i in range(n_reviews):
    nb = chosen_nb[i]
    site = chosen_site[i]
    site_type = sites.loc[sites.site_name == site, "site_type"].iloc[0]
    r = sentiment_roll[i]
    if r < 0.45:
        phrase = RNG.choice(positive_phrases)
    elif r < 0.75:
        phrase = RNG.choice(negative_phrases)
    else:
        phrase = RNG.choice(neutral_phrases)
    text = f"{site} {phrase}."
    records.append({"review_id": i, "site_name": site, "neighborhood": nb,
                     "site_type": site_type, "text": text})

reviews = pd.DataFrame(records)
reviews.head()

### How many reviews come from each neighborhood?
Raw review volume shapes which neighborhoods' opinions dominate any "what people are saying" summary.

In [ ]:
review_counts = reviews["neighborhood"].value_counts()

fig_volume, ax = plt.subplots(figsize=(8, 5))
review_counts.plot(kind="bar", ax=ax)
ax.set_title("Review Count by Neighborhood")
ax.set_xlabel("Neighborhood")
ax.set_ylabel("Number of reviews")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### What are people talking about, citywide?
TF-IDF scores words by how distinctive they are across reviews, which surfaces recurring themes better than raw word counts.

In [ ]:
# ▶ number of top keywords to display (also used in the Experiment section below)
TOP_N_KEYWORDS = 12

vectorizer = TfidfVectorizer(stop_words="english", max_features=200, min_df=3)
tfidf_matrix = vectorizer.fit_transform(reviews["text"])
scores = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
terms = vectorizer.get_feature_names_out()

top_keywords = pd.DataFrame({"keyword": terms, "tfidf_score": scores}).sort_values(
    "tfidf_score", ascending=False).head(TOP_N_KEYWORDS).reset_index(drop=True)
top_keywords

### How positive or negative is the feedback?
A simple, transparent keyword-count sentiment score (not a black-box model) lets us see exactly why a review was scored positive or negative -- important when the score will inform a planning decision.

In [ ]:
positive_words = {"clean", "great", "shade", "benches", "safe", "easy", "bike",
                   "frequent", "reliable", "accessible", "strollers", "wheelchairs"}
negative_words = {"lighting", "unsafe", "dark", "crowded", "noisy", "broken", "litter",
                   "unreliable", "delayed", "hard", "car"}


def lexicon_sentiment(text):
    words = text.lower().replace(".", "").split()
    pos = sum(w in positive_words for w in words)
    neg = sum(w in negative_words for w in words)
    total = max(pos + neg, 1)
    return (pos - neg) / total


reviews["sentiment_score"] = reviews["text"].apply(lexicon_sentiment)
avg_sentiment_by_type = reviews.groupby("site_type")["sentiment_score"].mean().sort_values()

fig_sentiment, ax = plt.subplots(figsize=(7, 5))
avg_sentiment_by_type.plot(kind="bar", ax=ax, color="tab:green")
ax.set_title("Average Review Sentiment by Site Type")
ax.set_xlabel("Site type")
ax.set_ylabel("Sentiment score (-1 negative to +1 positive)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Responsible AI check: whose voices are over- or under-represented?
Compare each neighborhood's *share of reviews* to its *share of city population*. A theme summary built only from review volume will overweight whichever neighborhood reviews the most, not whichever neighborhood has the most people or the most need.

In [ ]:
review_share = (review_counts / review_counts.sum()).rename("review_share")
pop_share_series = pd.Series(pop_share, name="population_share")
representation = pd.concat([review_share, pop_share_series], axis=1)
representation["representation_ratio"] = representation["review_share"] / representation["population_share"]
representation = representation.sort_values("representation_ratio")

fig_representation, ax = plt.subplots(figsize=(8, 5))
representation[["review_share", "population_share"]].plot(kind="bar", ax=ax)
ax.axhline(1 / len(representation), color="gray", linestyle=":", linewidth=1)
ax.set_title("Review Share vs. Population Share, by Neighborhood")
ax.set_xlabel("Neighborhood")
ax.set_ylabel("Share")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(representation.round(3))
most_underrepresented = representation["representation_ratio"].idxmin()
print(f"\nMost under-represented in the review data relative to population: {most_underrepresented}")

## Experiment

Try changing the parameters marked `# ▶ CHANGE ME` in the next cell(s) and re-run. Specifically, try:

1. Change `TOP_N_KEYWORDS` to show more or fewer themes.
2. Change `FOCUS_NEIGHBORHOOD` below to see that neighborhood's own top themes.
3. Lower `MIN_DF_EXPERIMENT` to let rarer words count as keywords -- does the theme list get noisier?


In [ ]:
# ▶ CHANGE ME: which neighborhood's themes to inspect
FOCUS_NEIGHBORHOOD = "Southside"  # try "Old Town", "Uptown", "Millbrook", ...
# ▶ CHANGE ME: minimum number of reviews a word must appear in in order to count
MIN_DF_EXPERIMENT = 2

focus_reviews = reviews[reviews["neighborhood"] == FOCUS_NEIGHBORHOOD]
focus_vectorizer = TfidfVectorizer(stop_words="english", min_df=MIN_DF_EXPERIMENT)
focus_matrix = focus_vectorizer.fit_transform(focus_reviews["text"])
focus_scores = np.asarray(focus_matrix.sum(axis=0)).flatten()
focus_terms = focus_vectorizer.get_feature_names_out()
focus_top = pd.DataFrame({"keyword": focus_terms, "tfidf_score": focus_scores}).sort_values(
    "tfidf_score", ascending=False).head(8)
print(f"Top themes in {FOCUS_NEIGHBORHOOD}:")
focus_top

## Artifact: Top-themes table + bias note

In [ ]:
bias_note = (
    f"Review volume is not proportional to population: {most_underrepresented} is the most "
    f"under-represented neighborhood in this review dataset relative to its population share "
    f"(representation ratio = {representation.loc[most_underrepresented, 'representation_ratio']:.2f}, "
    f"where 1.0 would mean reviews match population exactly). Any theme summary built from this data "
    f"should be labeled as reflecting reviewers, not all residents, and should be supplemented with "
    f"outreach in under-represented neighborhoods before being used to set priorities."
)
print(bias_note)

top_keywords.to_csv("lab1_top_themes.csv", index=False)
with open("lab1_bias_note.txt", "w") as f:
    f.write(bias_note)
print("\nSaved artifacts: lab1_top_themes.csv, lab1_bias_note.txt")

## Reflect (answer in your own words — this is part of your mini-task)

1. What did the tool assume?
2. Who is missing from this data?
3. What would change your recommendation?
4. What must a human verify before this is used?
